In [ ]:
%config InlineBackend.figure_format = 'retina'

import json
import matplotlib.pyplot as plt
import numpy as np
import re

In [ ]:
base_path = "../../logs/frames_llamacpp_qwen3_1.7b"
regex = r"logprob=(-?\d+\.\d+(?:[eE][-+]?\d+)?)"
all_logprobs = []
for i in range(150):
    with open(f"{base_path}/{i}/run_0/raw/trace.json", "r") as f:
        trace = json.load(f)
        trace = [t for t in trace if t["name"] == "ActionStep" and "logprob=" in t["attributes"]["output.value"]]
        trace = sorted(trace, key=lambda t: t["start_time"])
        logprobs = []
        for t in trace:
            v = json.loads(t["attributes"]["output.value"])
            logprobs.append([
                float(re.search(regex, l).group(1)) for l in v["model_output_message"]["raw"]["logprobs"]
            ])
        all_logprobs.append(logprobs)

In [ ]:
def plot_step_logprobs_stats(func):
    step_logprobs_data = [[] for _ in range(11)]
    for logprobs in all_logprobs:
        for step, step_logprobs in enumerate(logprobs):
            step_logprobs_data[step].append(func(step_logprobs))

    plt.figure(figsize=(8, 5))
    plt.boxplot(
        step_logprobs_data,
        positions=list(range(1, 12)),
        patch_artist=True
    )
    plt.xlabel("Step")
    func_name = func.__name__.split(".")[-1]
    func_name = func_name[0].upper() + func_name[1:]
    plt.ylabel(f"{func_name} logprobs")
    plt.title(f"{func_name} logprobs vs Step")
    plt.show()

In [ ]:
plot_step_logprobs_stats(min)

In [ ]:
plot_step_logprobs_stats(np.mean)

In [ ]:
plot_step_logprobs_stats(sum)

In [ ]:
min_logprobs_threshold = -1.25
cascade_steps = [None] * len(all_logprobs)
for i, logprobs in enumerate(all_logprobs):
    for step, step_logprobs in enumerate(logprobs):
        if min(step_logprobs) < min_logprobs_threshold:
            cascade_steps[i] = step
            break

for step in [None] + list(range(10)):
    num_cascade = sum(1 for s in cascade_steps if s == step)
    print(f"Cascade at step {step}: {num_cascade} ({round(num_cascade / len(cascade_steps) * 100, 2)} %)")